In [9]:
# ----- Importações -----

import pandas as pd
import requests
import io
import os
import time

In [10]:
# ----- Variáveis Globais -----
# Pasta
pasta_dados = 'dados_dengue_infodengue'
pasta_long = f'{pasta_dados}/long'
pasta_wide = f'{pasta_dados}/wide'

# URL da API do InfoDengue 
url_api = 'https://info.dengue.mat.br/api/alertcity'

# Dicionário com todos os municípios do estado do Pará, com seus nomes e códigos IBGE, para os quais queremos coletar os dados de dengue
municipios_ibge = {
    "1500107": "Abaetetuba",
    "1500131": "Abel Figueiredo",
    "1500206": "Acará",
    "1500305": "Afuá",
    "1500347": "Água Azul do Norte",
    "1500404": "Alenquer",
    "1500503": "Almeirim",
    "1500602": "Altamira",
    "1500701": "Anajás",
    "1500800": "Ananindeua",
    "1500859": "Anapu",
    "1500909": "Augusto Corrêa",
    "1500958": "Aurora do Pará",
    "1501006": "Aveiro",
    "1501105": "Bagre",
    "1501204": "Baião",
    "1501253": "Bannach",
    "1501303": "Barcarena",
    "1501402": "Belém",
    "1501451": "Belterra",
    "1501501": "Benevides",
    "1501576": "Bom Jesus do Tocantins",
    "1501600": "Bonito",
    "1501709": "Bragança",
    "1501725": "Brasil Novo",
    "1501758": "Brejo Grande do Araguaia",
    "1501782": "Breu Branco",
    "1501808": "Breves",
    "1501907": "Bujaru",
    "1502004": "Cachoeira do Arari",
    "1501956": "Cachoeira do Piriá",
    "1502103": "Cametá",
    "1502152": "Canaã dos Carajás",
    "1502202": "Capanema",
    "1502301": "Capitão Poço",
    "1502400": "Castanhal",
    "1502509": "Chaves",
    "1502608": "Colares",
    "1502707": "Conceição do Araguaia",
    "1502756": "Concórdia do Pará",
    "1502764": "Cumaru do Norte",
    "1502772": "Curionópolis",
    "1502806": "Curralinho",
    "1502855": "Curuá",
    "1502905": "Curuçá",
    "1502939": "Dom Eliseu",
    "1502954": "Eldorado do Carajás",
    "1503002": "Faro",
    "1503044": "Floresta do Araguaia",
    "1503077": "Garrafão do Norte",
    "1503093": "Goianésia do Pará",
    "1503101": "Gurupá",
    "1503200": "Igarapé-Açu",
    "1503309": "Igarapé-Miri",
    "1503408": "Inhangapi",
    "1503457": "Ipixuna do Pará",
    "1503507": "Irituia",
    "1503606": "Itaituba",
    "1503705": "Itupiranga",
    "1503754": "Jacareacanga",
    "1503804": "Jacundá",
    "1503903": "Juruti",
    "1504000": "Limoeiro do Ajuru",
    "1504059": "Mãe do Rio",
    "1504109": "Magalhães Barata",
    "1504208": "Marabá",
    "1504307": "Maracanã",
    "1504406": "Marapanim",
    "1504422": "Marituba",
    "1504455": "Medicilândia",
    "1504505": "Melgaço",
    "1504604": "Mocajuba",
    "1504703": "Moju",
    "1504752": "Mojuí dos Campos",
    "1504802": "Monte Alegre",
    "1504901": "Muaná",
    "1504950": "Nova Esperança do Piriá",
    "1504976": "Nova Ipixuna",
    "1505007": "Nova Timboteua",
    "1505031": "Novo Progresso",
    "1505064": "Novo Repartimento",
    "1505106": "Óbidos",
    "1505205": "Oeiras do Pará",
    "1505304": "Oriximiná",
    "1505403": "Ourém",
    "1505437": "Ourilândia do Norte",
    "1505486": "Pacajá",
    "1505494": "Palestina do Pará",
    "1505502": "Paragominas",
    "1505536": "Parauapebas",
    "1505551": "Pau D'Arco",
    "1505601": "Peixe-Boi",
    "1505635": "Piçarra",
    "1505650": "Placas",
    "1505700": "Ponta de Pedras",
    "1505809": "Portel",
    "1505908": "Porto de Moz",
    "1506005": "Prainha",
    "1506104": "Primavera",
    "1506112": "Quatipuru",
    "1506138": "Redenção",
    "1506161": "Rio Maria",
    "1506187": "Rondon do Pará",
    "1506195": "Rurópolis",
    "1506203": "Salinópolis",
    "1506302": "Salvaterra",
    "1506351": "Santa Bárbara do Pará",
    "1506401": "Santa Cruz do Arari",
    "1506500": "Santa Izabel do Pará",
    "1506559": "Santa Luzia do Pará",
    "1506583": "Santa Maria das Barreiras",
    "1506609": "Santa Maria do Pará",
    "1506708": "Santana do Araguaia",
    "1506807": "Santarém",
    "1506906": "Santarém Novo",
    "1507003": "Santo Antônio do Tauá",
    "1507102": "São Caetano de Odivelas",
    "1507151": "São Domingos do Araguaia",
    "1507201": "São Domingos do Capim",
    "1507300": "São Félix do Xingu",
    "1507409": "São Francisco do Pará",
    "1507458": "São Geraldo do Araguaia",
    "1507466": "São João da Ponta",
    "1507474": "São João de Pirabas",
    "1507508": "São João do Araguaia",
    "1507607": "São Miguel do Guamá",
    "1507706": "São Sebastião da Boa Vista",
    "1507755": "Sapucaia",
    "1507805": "Senador José Porfírio",
    "1507904": "Soure",
    "1507953": "Tailândia",
    "1507961": "Terra Alta",
    "1507979": "Terra Santa",
    "1508001": "Tomé-Açu",
    "1508035": "Tracuateua",
    "1508050": "Trairão",
    "1508084": "Tucumã",
    "1508100": "Tucuruí",
    "1508126": "Ulianópolis",
    "1508159": "Uruará",
    "1508209": "Vigia",
    "1508308": "Viseu",
    "1508357": "Vitória do Xingu",
    "1508407": "Xinguara"
}

# Passando os parâmetros da consulta como dicionário para o requests, que irá convertê-los em query string automaticamente
params = {
    'geocode': '',
    'disease': 'dengue',
    'format': 'csv',
    'ew_start': 1,
    'ew_end': 53,
    'ey_start': 2000,
    'ey_end': 2026
}

print(f"[INFO] Total de municípios: {len(municipios_ibge)}")
print(f"[INFO] Os parâmetros são: {params}")

[INFO] Total de municípios: 144
[INFO] Os parâmetros são: {'geocode': '', 'disease': 'dengue', 'format': 'csv', 'ew_start': 1, 'ew_end': 53, 'ey_start': 2000, 'ey_end': 2026}


In [ ]:
# Criar pastas para organização
os.makedirs(f'{pasta_long}', exist_ok=True)
os.makedirs(f'{pasta_wide}', exist_ok=True)

# Medir o tempo total de execução
start_time = time.perf_counter()

# ----- Coleta, Transformação e Armazenamento -----
for cod_ibge, nome_municipio in municipios_ibge.items():
    params['geocode'] = cod_ibge
    
    try:
        # Timeout para 15s para evitar falhas em municípios com muitos dados ou conexões lentas
        response = requests.get(url_api, params=params, timeout=15) 
        response.encoding = 'utf-8'
        
        if response.status_code == 200:
            # Lemos o CSV diretamente da resposta usando o pandas, que é mais eficiente do que salvar e ler do disco
            df = pd.read_csv(io.StringIO(response.text))
            
            # Adicionamos metadados
            df['municipio_nome_limpo'] = nome_municipio
            df['municipio_cod_ibge'] = cod_ibge
            
            # --- VERSÃO NORMAL ---
            caminho_normal = f'{pasta_wide}/dados_wide_{nome_municipio}_{cod_ibge}.csv'
            df.to_csv(caminho_normal, index=False)
            
            # --- VERSÃO MELT ---
            # Definimos as colunas que identificam a linha (não serão transformadas)
            id_vars = [
                'data_iniSE', 'SE', 'municipio_nome_limpo', 'municipio_cod_ibge', 
                'versao_modelo', 'pop', 'nivel', 'id'
            ]

            # Todas as colunas que não estão em id_vars serão transformadas de wide para long
            df_melt = df.melt(id_vars=id_vars, var_name='indicador', value_name='valor')
            
            caminho_melt = f'{pasta_long}/dados_long_{nome_municipio}_{cod_ibge}.csv'
            df_melt.to_csv(caminho_melt, index=False)
            
            print(f"[INFO] Processado: {nome_municipio}")
        else:
            print(f"[ERRO] Falha {nome_municipio}: Status {response.status_code}")
            
    except Exception as e:
        print(f"[ERRO] Não foi possível processar {nome_municipio}: {e}")

end_time = time.perf_counter()
print(f"[INFO] Tempo total de execução: {end_time - start_time:.2f} segundos")

[INFO] Processado: Abaetetuba
[INFO] Processado: Abel Figueiredo
[INFO] Processado: Acará
[INFO] Processado: Afuá
[INFO] Processado: Água Azul do Norte
[INFO] Processado: Alenquer
[INFO] Processado: Almeirim
[INFO] Processado: Altamira
[INFO] Processado: Anajás
[INFO] Processado: Ananindeua
[INFO] Processado: Anapu
[INFO] Processado: Augusto Corrêa
[INFO] Processado: Aurora do Pará
[INFO] Processado: Aveiro
[INFO] Processado: Bagre
[INFO] Processado: Baião
[INFO] Processado: Bannach
[INFO] Processado: Barcarena
[INFO] Processado: Belém
[INFO] Processado: Belterra
[INFO] Processado: Benevides
[INFO] Processado: Bom Jesus do Tocantins
[INFO] Processado: Bonito
[INFO] Processado: Bragança
[INFO] Processado: Brasil Novo
[INFO] Processado: Brejo Grande do Araguaia
[INFO] Processado: Breu Branco
[INFO] Processado: Breves
[INFO] Processado: Bujaru
[INFO] Processado: Cachoeira do Arari
[INFO] Processado: Cachoeira do Piriá
[INFO] Processado: Cametá
[INFO] Processado: Canaã dos Carajás
[INFO] P